# LeetCode 955 — Delete Columns to Make Sorted II bằng Python

Link đề: https://leetcode.com/problems/delete-columns-to-make-sorted-ii/ · Độ khó: **Medium** · Chủ đề: Array (mảng), String (chuỗi), Greedy (tham lam)

Skeleton LeetCode:

```python
class Solution:
    def minDeletionSize(self, strs: List[str]) -> int:
        ...
```

Ghi chú: ta được **xóa một số cột** (cùng chỉ số ở mọi chuỗi) để các chuỗi còn lại **sắp xếp theo thứ tự từ điển** `strs[0] <= strs[1] <= ... <= strs[n - 1]`; cần số cột xóa ít nhất. Khác với bài 944 (mỗi **cột** phải không giảm), ở đây điều kiện là về **các chuỗi** theo thứ tự từ điển: một cột sau **không cần** đúng thứ tự nếu cặp chuỗi kề nhau đã được **phân định** (một chuỗi đã nhỏ hơn hẳn chuỗi kia) ở một cột đã giữ phía trước. Tham lam theo từng cột từ trái sang phải với mảng "các cặp đã phân định".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn tập cột xóa nhỏ nhất để các chuỗi còn lại tăng (không giảm) theo thứ tự từ điển.
2. Nắm các công cụ Python cần thiết: `zip(*strs)` để lấy cột, so sánh chuỗi (`<=`), mảng cờ `resolved[i]` cho các cặp kề, và liệt kê tập cột giữ bằng `itertools.combinations`/bitmask.
3. Hiểu nguyên lý: so sánh từ điển được quyết định bởi **vị trí đầu tiên khác nhau**; sau khi một cột đã giữ làm cặp `(i, i + 1)` có `s[i][c] < s[i + 1][c]` thì cặp đó **đã phân định** và mọi cột sau bị bỏ qua cho cặp ấy.
4. Đi qua 4 cách: thử mọi tập cột giữ (brute), tham lam theo cột với cờ phân định (cách nộp chính), tham lam bằng dựng lại chuỗi đang giữ, và quy hoạch động theo tập cặp đã phân định (độc lập với giả thiết tham lam); đối chiếu và kiểm tra chéo.
5. Tránh các lỗi kinh điển: nhầm với bài 944 (xét từng cột), kiểm tra mọi cặp ở mọi cột (không dùng cờ phân định), coi cặp bằng nhau là đã phân định, và cập nhật cờ trước khi quyết định xóa cột.

## 2. Đề bài & ví dụ

Cho mảng `n` chuỗi `strs` cùng độ dài.

Ta có thể chọn **chỉ số xóa** bất kỳ và xóa **mọi ký tự ở các chỉ số đó** trong từng chuỗi. Ví dụ `strs = ["abcdef", "uvwxyz"]` với chỉ số xóa `{0, 2, 3}` cho `["bef", "vyz"]`.

Giả sử ta chọn tập chỉ số xóa `answer` sao cho mảng cuối cùng có các phần tử **theo thứ tự từ điển** (`strs[0] <= strs[1] <= ... <= strs[n - 1]`). Trả về **giá trị nhỏ nhất** của `answer.length`.

**Ví dụ 1:** `strs = ["ca","bb","ac"]` -> `1` (xóa cột `0` còn `["a","b","c"]`).

**Ví dụ 2:** `strs = ["xc","yb","za"]` -> `0` (đã sắp xếp; các hàng không nhất thiết tăng theo ký tự).

**Ví dụ 3:** `strs = ["zyx","wvu","tsr"]` -> `3` (xóa mọi cột).

**Ràng buộc:**

- `n == strs.length`, `1 <= n <= 100`
- `1 <= strs[i].length <= 100`
- `strs[i]` gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
from itertools import combinations
import random, time

examples = [
    (["ca", "bb", "ac"], 1),
    (["xc", "yb", "za"], 0),
    (["zyx", "wvu", "tsr"], 3),
]
for strs, expected in examples:
    print(f"strs={strs} kỳ vọng={expected}")

strs=['ca', 'bb', 'ac'] kỳ vọng=1
strs=['xc', 'yb', 'za'] kỳ vọng=0
strs=['zyx', 'wvu', 'tsr'] kỳ vọng=3


## 3. Kiến thức nền

### 3.1. So sánh từ điển và vị trí khác nhau đầu tiên

`a <= b` (từ điển) ⇔ `a == b`, hoặc tại **vị trí đầu tiên** `c` mà `a[c] != b[c]` ta có `a[c] < b[c]`. Các vị trí sau đó **không còn ảnh hưởng**.

In [2]:
for a, b in (("abz", "acb"), ("abc", "abc"), ("abd", "abc"), ("a", "ab")):
    print(f"{a!r} <= {b!r}: {a <= b}")

'abz' <= 'acb': True
'abc' <= 'abc': True
'abd' <= 'abc': False
'a' <= 'ab': True


### 3.2. Lấy cột bằng `zip(*strs)`

`zip(*strs)` chuyển hàng thành cột: cột `c` là bộ `(strs[0][c], strs[1][c], ..., strs[n-1][c])`.

In [3]:
strs = ["ca", "bb", "ac"]
print(list(zip(*strs)))

[('c', 'b', 'a'), ('a', 'b', 'c')]


### 3.3. Cặp đã phân định

Sau khi quyết định giữ một số cột, xét cặp kề `(i, i + 1)`: nếu có một cột đã giữ với `strs[i][c] < strs[i + 1][c]` (và ở mọi cột giữ **trước** nó hai ký tự bằng nhau) thì chuỗi `i` đã **nhỏ hơn hẳn** chuỗi `i + 1`: mọi cột giữ sau đó **không cần** thỏa gì cho cặp này. Cặp chưa phân định (mọi cột giữ cho tới giờ đều bằng nhau) bắt buộc `strs[i][c] <= strs[i + 1][c]` ở cột giữ mới.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `["ca", "bb", "ac"]`; cặp `(0, 1)` và `(1, 2)`; ban đầu chưa phân định.

- Cột `0`: `c, b, a`. Cặp `(0,1)`: `c > b` vi phạm ⇒ **xóa** cột `0`.
- Cột `1`: `a, b, c`. Cặp `(0,1)`: `a < b` (phân định); cặp `(1,2)`: `b < c` (phân định). Giữ cột `1`.

Xóa `1` cột. Ví dụ 2: `["xc","yb","za"]`: cột `0`: `x < y`, `y < z` cả hai phân định ⇒ giữ; cột `1`: `c, b, a` vi phạm nhưng mọi cặp đã phân định nên **được bỏ qua**: giữ ⇒ `0` cột xóa.

## 5. Cách 1 — Brute Force: thử mọi tập cột giữ

**Ý tưởng.** Với mỗi tập con cột giữ (`2^m`), tạo các chuỗi rút gọn và kiểm tra `strs[i] <= strs[i + 1]` với mọi `i`; lấy tập giữ lớn nhất (xóa ít nhất). Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [4]:
def min_deletion_brute(strs: List[str]) -> int:
    m = len(strs[0])
    best_keep = 0
    for mask in range(1 << m):
        keep = [c for c in range(m) if mask >> c & 1]
        reduced = ["".join(s[c] for c in keep) for s in strs]
        if all(reduced[i] <= reduced[i + 1] for i in range(len(reduced) - 1)):
            best_keep = max(best_keep, len(keep))
    return m - best_keep

for strs, expected in examples:
    got = min_deletion_brute(strs)
    print(strs, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['ca', 'bb', 'ac'] -> 1 OK
['xc', 'yb', 'za'] -> 0 OK
['zyx', 'wvu', 'tsr'] -> 3 OK


**Độ phức tạp và nút thắt.** `O(2^m * n * m)`: với `m = 100` là không thể. Xem thời gian:

In [5]:
rng = random.Random(955)
for m in (6, 8, 10, 12, 14):
    strs = ["".join(rng.choice("abc") for _ in range(m)) for _ in range(5)]
    t0 = time.perf_counter()
    min_deletion_brute(strs)
    print(f"m={m:2}: {1000 * (time.perf_counter() - t0):9.1f} ms")

m= 6:       0.2 ms
m= 8:       0.7 ms
m=10:       3.1 ms
m=12:      14.2 ms
m=14:      59.1 ms


Thời gian tăng gấp đôi mỗi cột. Nút thắt: ta xét cả những tập cột hiển nhiên không cần; thực ra quyết định **giữ hay xóa từng cột** chỉ phụ thuộc vào các cặp **chưa phân định**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam theo cột từ trái sang phải.** Với cột `c`: nếu có một cặp kề **chưa phân định** `(i, i + 1)` mà `strs[i][c] > strs[i + 1][c]` thì **bắt buộc xóa** cột `c` (giữ nó sẽ khiến `strs[i] > strs[i+1]`). Ngược lại **giữ** cột `c` (giữ thêm một cột chỉ giúp: các cặp có `<` được phân định, các cặp `=` vẫn chưa) và cập nhật các cặp có `strs[i][c] < strs[i + 1][c]` thành đã phân định.

Giữ khi có thể là tối ưu vì cột nào hợp lệ để giữ thì luôn có thể giữ mà không làm các cột sau **khó giữ hơn** (chứng minh ở mục 10).

| Cách | Ý tưởng |
|---|---|
| 2 | Tham lam với cờ `resolved[i]` (nộp) |
| 3 | Tham lam: thử thêm cột vào các chuỗi đang giữ và kiểm tra sắp xếp |
| 4 | Quy hoạch động trên **tập cặp đã phân định** (kiểm chứng độc lập) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam với cờ phân định (cách nộp chính)

**Giải thích từng đoạn.**

- `resolved = [False] * (n - 1)`: `resolved[i]` đúng khi cặp `(i, i + 1)` đã phân định.
- Với mỗi cột `c`:
  - nếu tồn tại `i` với `not resolved[i]` và `strs[i][c] > strs[i + 1][c]` thì `deleted += 1` và **không cập nhật** `resolved`;
  - ngược lại cập nhật `resolved[i] = True` với mọi `i` có `strs[i][c] < strs[i + 1][c]` (kể cả đã phân định: không đổi).
- Trả `deleted`.

In [6]:
def min_deletion_greedy(strs: List[str]) -> int:
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    deleted = 0
    for c in range(m):
        if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
            deleted += 1
            continue
        for i in range(n - 1):
            if strs[i][c] < strs[i + 1][c]:
                resolved[i] = True
    return deleted

for strs, expected in examples:
    got = min_deletion_greedy(strs)
    print(strs, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['ca', 'bb', 'ac'] -> 1 OK
['xc', 'yb', 'za'] -> 0 OK
['zyx', 'wvu', 'tsr'] -> 3 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def minDeletionSize(self, strs: List[str]) -> int:
        n, m = len(strs), len(strs[0])
        resolved = [False] * (n - 1)
        deleted = 0
        for c in range(m):
            if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
                deleted += 1
                continue
            for i in range(n - 1):
                if strs[i][c] < strs[i + 1][c]:
                    resolved[i] = True
        return deleted

sol_greedy = Solution().minDeletionSize
assert [sol_greedy(list(s)) for s, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n * m)` thời gian (mỗi cột quét `n - 1` cặp tối đa hai lần), `O(n)` bộ nhớ.

---

### 7.2. Cách 3 — Dựng lại chuỗi đang giữ

**Ý tưởng khác.** Không dùng cờ: duy trì `kept = [""] * n` (các chuỗi rút gọn đang giữ). Với cột `c`: tạo `candidate = [kept[i] + strs[i][c] for i in range(n)]`; nếu `candidate` đã sắp xếp (`all(candidate[i] <= candidate[i + 1])`) thì giữ (`kept = candidate`), ngược lại xóa. Thuật toán hiển nhiên đúng với định nghĩa của tham lam "giữ nếu không phá vỡ thứ tự", nhưng tốn `O(n * m^2)` do so sánh chuỗi dài dần.

In [8]:
def min_deletion_rebuild(strs: List[str]) -> int:
    n, m = len(strs), len(strs[0])
    kept = [""] * n
    deleted = 0
    for c in range(m):
        candidate = [kept[i] + strs[i][c] for i in range(n)]
        if all(candidate[i] <= candidate[i + 1] for i in range(n - 1)):
            kept = candidate
        else:
            deleted += 1
    return deleted

for strs, expected in examples:
    got = min_deletion_rebuild(strs)
    print(strs, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['ca', 'bb', 'ac'] -> 1 OK
['xc', 'yb', 'za'] -> 0 OK
['zyx', 'wvu', 'tsr'] -> 3 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def minDeletionSize(self, strs: List[str]) -> int:
        n, m = len(strs), len(strs[0])
        kept = [""] * n
        deleted = 0
        for c in range(m):
            candidate = [kept[i] + strs[i][c] for i in range(n)]
            if all(candidate[i] <= candidate[i + 1] for i in range(n - 1)):
                kept = candidate
            else:
                deleted += 1
        return deleted

sol_rebuild = Solution().minDeletionSize
assert [sol_rebuild(list(s)) for s, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n * m^2)` thời gian (so sánh các chuỗi dài tới `m`), `O(n * m)` bộ nhớ.

---

### 7.3. Cách 4 — Quy hoạch động trên tập cặp đã phân định

**Ý tưởng khác.** Không dùng giả thiết tham lam: trạng thái là **tập các cặp kề đã phân định** (bitmask `n - 1` bit), giá trị là **số cột giữ lớn nhất**. Với mỗi cột:

- **xóa** cột: trạng thái giữ nguyên;
- **giữ** cột: hợp lệ khi mọi cặp **chưa** phân định có `strs[i][c] <= strs[i + 1][c]`; trạng thái mới thêm các cặp có `<`.

Đáp án `m - max(giữ)`. Số trạng thái `<= 2^(n-1)`, dùng cho `n` nhỏ (kiểm chứng độc lập tính tối ưu của tham lam).

In [10]:
def min_deletion_dp(strs: List[str]) -> int:
    n, m = len(strs), len(strs[0])
    dp = {0: 0}                                          # mask các cặp đã phân định -> số cột giữ lớn nhất
    for c in range(m):
        new = dict(dp)                                   # xóa cột c
        for mask, kept in dp.items():
            ok = True
            add = 0
            for i in range(n - 1):
                a, b = strs[i][c], strs[i + 1][c]
                if not (mask >> i & 1) and a > b:
                    ok = False
                    break
                if a < b:
                    add |= 1 << i
            if ok:
                nm = mask | add
                if new.get(nm, -1) < kept + 1:
                    new[nm] = kept + 1
        dp = new
    return m - max(dp.values())

for strs, expected in examples:
    got = min_deletion_dp(strs)
    print(strs, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['ca', 'bb', 'ac'] -> 1 OK
['xc', 'yb', 'za'] -> 0 OK
['zyx', 'wvu', 'tsr'] -> 3 OK


**Code nộp LeetCode (Cách 4):** chỉ dùng được khi `n` nhỏ (số trạng thái mũ theo `n`); không nộp cho `n = 100`. Cách này dùng để kiểm chứng.

**Độ phức tạp (Cách 4):** `O(m * 2^(n-1) * n)` thời gian, `O(2^(n-1))` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 1: `["ca", "bb", "ac"]`

In [11]:
def trace_greedy(strs):
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    deleted = 0
    for c in range(m):
        col = [s[c] for s in strs]
        bad = [i for i in range(n - 1) if not resolved[i] and strs[i][c] > strs[i + 1][c]]
        if bad:
            deleted += 1
            print(f"cột {c} = {col}: cặp chưa phân định {bad} vi phạm -> XÓA (đã xóa {deleted}), resolved giữ {resolved}")
            continue
        newly = [i for i in range(n - 1) if strs[i][c] < strs[i + 1][c]]
        for i in newly:
            resolved[i] = True
        print(f"cột {c} = {col}: giữ; phân định thêm các cặp {newly} -> resolved = {resolved}")
    return deleted

assert trace_greedy(["ca", "bb", "ac"]) == 1

cột 0 = ['c', 'b', 'a']: cặp chưa phân định [0, 1] vi phạm -> XÓA (đã xóa 1), resolved giữ [False, False]
cột 1 = ['a', 'b', 'c']: giữ; phân định thêm các cặp [0, 1] -> resolved = [True, True]


### 8.2. Cách 2 cho ví dụ 2: các cột sau được bỏ qua

In [12]:
assert trace_greedy(["xc", "yb", "za"]) == 0

cột 0 = ['x', 'y', 'z']: giữ; phân định thêm các cặp [0, 1] -> resolved = [True, True]
cột 1 = ['c', 'b', 'a']: giữ; phân định thêm các cặp [] -> resolved = [True, True]


Ở cột `1` (`c, b, a`) có `c > b` nhưng cặp `(0,1)` **đã phân định** ở cột `0` (`x < y`), nên không cần quan tâm.

### 8.3. Cách 4 (DP) cho ví dụ 1

Hai cặp ⇒ bitmask `00..11`. Cột 0 (`c,b,a`): giữ không hợp lệ ở trạng thái `00` (cặp 0: `c > b`); chỉ có "xóa": `dp = {0: 0}`. Cột 1 (`a,b,c`): giữ hợp lệ, `a < b` và `b < c` thêm cả hai bit: `dp = {0: 0, 3: 1}`. Max giữ `1`; xóa `2 - 1 = 1`.

In [13]:
strs = ["ca", "bb", "ac"]
n, m = 3, 2
dp = {0: 0}
for c in range(m):
    new = dict(dp)
    for mask, kept in dp.items():
        ok, add = True, 0
        for i in range(n - 1):
            a, b = strs[i][c], strs[i + 1][c]
            if not (mask >> i & 1) and a > b:
                ok = False
                break
            if a < b:
                add |= 1 << i
        if ok:
            new[mask | add] = max(new.get(mask | add, -1), kept + 1)
    dp = new
    print(f"sau cột {c}: dp = {dp}")
assert m - max(dp.values()) == 1

sau cột 0: dp = {0: 0}
sau cột 1: dp = {0: 0, 3: 1}


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (tham lam với cờ phân định): `O(n * m)`.

```python
class Solution:
    def minDeletionSize(self, strs: List[str]) -> int:
        n, m = len(strs), len(strs[0])
        resolved = [False] * (n - 1)
        deleted = 0
        for c in range(m):
            if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
                deleted += 1
                continue
            for i in range(n - 1):
                if strs[i][c] < strs[i + 1][c]:
                    resolved[i] = True
        return deleted
```

In [14]:
class Solution:
    def minDeletionSize(self, strs: List[str]) -> int:
        n, m = len(strs), len(strs[0])
        resolved = [False] * (n - 1)
        deleted = 0
        for c in range(m):
            if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
                deleted += 1
                continue
            for i in range(n - 1):
                if strs[i][c] < strs[i + 1][c]:
                    resolved[i] = True
        return deleted

sol = Solution()
for strs, expected in examples:
    got = sol.minDeletionSize(list(strs))
    print(strs, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

['ca', 'bb', 'ac'] -> 1 OK
['xc', 'yb', 'za'] -> 0 OK
['zyx', 'wvu', 'tsr'] -> 3 OK


## 10. Vì sao đúng

**(a) Điều kiện giữ một cột.** Xét tập cột giữ `K` (theo thứ tự tăng). Các chuỗi rút gọn được so sánh từ điển theo thứ tự `K`. Với cặp kề `(i, i + 1)`, `strs[i]|_K <= strs[i + 1]|_K` ⇔ hoặc hai chuỗi bằng nhau trên `K`, hoặc tại cột đầu tiên `c` trong `K` với hai ký tự khác nhau thì `strs[i][c] < strs[i + 1][c]`. Tức là với mỗi cột `c in K` mà cặp này **chưa phân định bởi các cột giữ nhỏ hơn**, cần `strs[i][c] <= strs[i + 1][c]`; và cột nào có `<` thì phân định cặp.

**(b) Tính tối ưu của tham lam.** Gọi `R(S)` là tập cặp đã phân định bởi tập cột giữ `S`. Một tập `K` là **hợp lệ** nếu mỗi cột `c in K` thỏa: với mọi cặp `i` chưa thuộc `R(K ∩ [0, c))` có `strs[i][c] <= strs[i + 1][c]` (đúng bằng điều kiện (a)). Gọi `G` là tập cột giữ của tham lam. Ta chứng minh bằng quy nạp theo `c`: với **mọi** tập hợp lệ `K`,

   - (i) `R(K ∩ [0, c]) ⊆ R(G ∩ [0, c])`,
   - (ii) `|K ∩ [0, c]| <= |G ∩ [0, c]|`.

   Cơ sở (`c = -1`): cả hai tập rỗng. Bước: xét cột `c`.
   - Nếu `K` **giữ** `c`: mọi cặp chưa thuộc `R(K ∩ [0, c))` thỏa `<=` tại `c`. Các cặp chưa thuộc `R(G ∩ [0, c))` cũng chưa thuộc `R(K ∩ [0, c))` (vì (i) cho `R(K ∩ [0, c)) ⊆ R(G ∩ [0, c))`), nên cũng thỏa `<=` tại `c`: tham lam cũng **giữ** `c`. Hai tập cùng thêm cột `c` nên (ii) được bảo toàn, và `R` của cả hai đều được thêm đúng tập cặp có `<` tại `c`, nên (i) vẫn đúng.
   - Nếu `K` **xóa** `c`: `R(K)` không đổi, còn `R(G)` chỉ lớn lên (hoặc giữ nguyên); (i) đúng; (ii) đúng vì `|G|` không giảm.
   Hệ quả: `|G| >= |K|` với mọi `K` hợp lệ, tức số cột xóa của tham lam `m - |G|` nhỏ nhất.

**Kiểm chứng bằng máy.** Hai oracle **độc lập với giả thiết tham lam**: (1) vét cạn mọi tập cột giữ (`m <= 12`); (2) **quy hoạch động theo tập cặp đã phân định** (Cách 4) cho `n <= 8` với `m` tới `40`, bao quát đúng mọi cách chọn giữ/xóa mà không giả định tham lam. Ngoài ra kiểm tra **bằng chứng**: tập cột xóa do thuật toán đưa ra (hàm `deleted_indices`) thực sự cho các chuỗi sắp xếp.

In [15]:
def deleted_indices(strs: List[str]) -> List[int]:
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    out = []
    for c in range(m):
        if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
            out.append(c)
            continue
        for i in range(n - 1):
            if strs[i][c] < strs[i + 1][c]:
                resolved[i] = True
    return out

def is_sorted_after_deleting(strs: List[str], deleted) -> bool:
    keep = [c for c in range(len(strs[0])) if c not in set(deleted)]
    reduced = ["".join(s[c] for c in keep) for s in strs]
    return all(reduced[i] <= reduced[i + 1] for i in range(len(reduced) - 1))

rng = random.Random(1955)
for _ in range(500):
    n, m = rng.randint(1, 5), rng.randint(1, 8)
    alphabet = "abcd"[:rng.randint(2, 4)]
    strs = ["".join(rng.choice(alphabet) for _ in range(m)) for _ in range(n)]
    b = min_deletion_brute(strs)
    assert b == min_deletion_dp(strs), strs
    d = deleted_indices(strs)
    assert len(d) == b and is_sorted_after_deleting(strs, d), (strs, d, b)
print("vét cạn, DP trên tập cặp phân định và lời giải tham lam (kèm bằng chứng) khớp nhau trên 500 ca")

vét cạn, DP trên tập cặp phân định và lời giải tham lam (kèm bằng chứng) khớp nhau trên 500 ca


## 11. Time / Space Complexity

Gọi `n` là số chuỗi, `m` độ dài mỗi chuỗi.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập cột | `O(2^m * n * m)` | `O(n * m)` |
| 2. Tham lam + cờ | `O(n * m)` | `O(n)` |
| 3. Dựng lại chuỗi | `O(n * m^2)` | `O(n * m)` |
| 4. DP tập cặp | `O(m * 2^(n-1) * n)` | `O(2^(n-1))` |

Đo thử với `n = 100`, `m = 100` (cỡ tối đa):

In [16]:
rng = random.Random(11)
n = m = 100
strs = sorted("".join(rng.choice("abc") for _ in range(m)) for _ in range(n))
strs2 = ["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(m)) for _ in range(n)]
for label, data in (("đã sắp xếp (0 cột xóa)", strs), ("ngẫu nhiên", strs2)):
    ref = sol_greedy(list(data))
    for name, fn in (("Cách 2", sol_greedy), ("Cách 3", sol_rebuild), ("nộp", Solution().minDeletionSize)):
        t0 = time.perf_counter()
        got = fn(list(data))
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, name
        print(f"{label:24} {name:7} n=m=100: {dt:8.2f} ms (đáp án {got})")

đã sắp xếp (0 cột xóa)   Cách 2  n=m=100:     1.47 ms (đáp án 0)
đã sắp xếp (0 cột xóa)   Cách 3  n=m=100:     2.20 ms (đáp án 0)
đã sắp xếp (0 cột xóa)   nộp     n=m=100:     1.79 ms (đáp án 0)
ngẫu nhiên               Cách 2  n=m=100:     0.08 ms (đáp án 100)
ngẫu nhiên               Cách 3  n=m=100:     0.70 ms (đáp án 100)
ngẫu nhiên               nộp     n=m=100:     0.06 ms (đáp án 100)


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [17]:
def solutions_list():
    return {
        "greedy": min_deletion_greedy,
        "rebuild": min_deletion_rebuild,
        "sol_greedy": sol_greedy,
        "sol_rebuild": sol_rebuild,
        "final": Solution().minDeletionSize,
    }

def run_all(strs, expected=None, brute_limit=12, dp_limit=8):
    n, m = len(strs), len(strs[0])
    if m <= brute_limit:
        ref = min_deletion_brute(strs)
    elif n <= dp_limit:
        ref = min_deletion_dp(strs)
    else:
        ref = min_deletion_greedy(strs)
    if n <= dp_limit:
        assert min_deletion_dp(strs) == ref, ("DP lệch", strs)
    for name, fn in solutions_list().items():
        assert fn(list(strs)) == ref, (name, strs)
    d = deleted_indices(strs)
    assert len(d) == ref and is_sorted_after_deleting(strs, d), (strs, d)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", strs, expected, ref)
    return ref

# 12.1 ví dụ đề
for strs, expected in examples:
    assert run_all(strs, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (["a"], 0),                                  # một chuỗi: luôn sắp xếp
    (["abc"], 0),
    (["b", "a"], 1),                             # một cột, đảo thứ tự: phải xóa
    (["a", "b"], 0),
    (["a", "a"], 0),                             # bằng nhau là hợp lệ
    (["ab", "ab"], 0),
    (["ba", "ab"], 1),                           # xóa cột 0 -> a, b; hoặc xóa cột 1 -> b, a (sai) nên phải xóa cột 0: 1
    (["ab", "ba"], 0),                           # cột 0: a < b phân định; cột 1 (b, a) bị bỏ qua
    (["xc", "yb", "za"], 0),
    (["ca", "bb", "ac"], 1),
    (["zyx", "wvu", "tsr"], 3),
    (["abx", "aby", "aab"], 2),                  # cột 0 bằng nhau; cột 1 (b,b,a) và cột 2 (x,y,b) đều phải xóa
    (["aa", "ab", "ba"], 0),
    (["bab", "abb", "aaa"], None),               # kiểm bằng oracle
    (["a" * 5] * 4, 0),                          # toàn bằng nhau
    (["z", "y", "x", "w"], 1),
]
for strs, expected in edge:
    run_all(strs, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 16 ca


In [18]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_column_sorted(strs):                     # nhầm với bài 944: mỗi cột phải không giảm từ trên xuống
    return sum(1 for col in zip(*strs) if any(col[i] > col[i + 1] for i in range(len(col) - 1)))

def wrong_check_all_pairs(strs):                   # kiểm tra mọi cặp ở mọi cột (không dùng cờ phân định)
    n, m = len(strs), len(strs[0])
    deleted = 0
    for c in range(m):
        if any(strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
            deleted += 1
    return deleted

def wrong_equal_is_resolved(strs):                 # coi cặp bằng nhau cũng là đã phân định (<=)
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    deleted = 0
    for c in range(m):
        if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
            deleted += 1
            continue
        for i in range(n - 1):
            if strs[i][c] <= strs[i + 1][c]:
                resolved[i] = True
    return deleted

def wrong_update_before_decide(strs):              # cập nhật cờ phân định trước khi quyết định xóa cột
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    deleted = 0
    for c in range(m):
        bad = any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1))
        for i in range(n - 1):
            if strs[i][c] < strs[i + 1][c]:
                resolved[i] = True                 # sai: cột bị xóa vẫn phân định cặp
        if bad:
            deleted += 1
    return deleted

def find_counter(wrong, tries=30000, seed=6):
    rng = random.Random(seed)
    for _ in range(tries):
        n, m = rng.randint(2, 4), rng.randint(2, 5)
        strs = ["".join(rng.choice("abc") for _ in range(m)) for _ in range(n)]
        if wrong(list(strs)) != min_deletion_brute(strs):
            return strs
    return None

counters = {}
for name, fn in (("nhầm với bài 944 (xét từng cột)", wrong_column_sorted), ("kiểm tra mọi cặp ở mọi cột", wrong_check_all_pairs),
                 ("coi cặp bằng nhau là đã phân định", wrong_equal_is_resolved), ("cập nhật cờ trước khi quyết định", wrong_update_before_decide)):
    strs = find_counter(fn)
    assert strs is not None, name
    counters[name] = strs
    print(f"{name:36} strs={strs}: sai -> {fn(list(strs))}, đúng -> {min_deletion_brute(strs)}")
    run_all(strs)
print("12.3 phản ví dụ: OK")

nhầm với bài 944 (xét từng cột)      strs=['abaac', 'cbbba', 'cbbbc', 'baacc']: sai -> 4, đúng -> 3
kiểm tra mọi cặp ở mọi cột           strs=['abaac', 'cbbba', 'cbbbc', 'baacc']: sai -> 4, đúng -> 3
coi cặp bằng nhau là đã phân định    strs=['babb', 'acbc', 'ccaa', 'cccc']: sai -> 1, đúng -> 3
cập nhật cờ trước khi quyết định     strs=['babb', 'acbc', 'ccaa', 'cccc']: sai -> 1, đúng -> 3
12.3 phản ví dụ: OK


In [19]:
# 12.4 ngẫu nhiên nhỏ: n <= 5, m <= 8, so với vét cạn và DP
rng = random.Random(4955)
for _ in range(1500):
    n, m = rng.randint(1, 5), rng.randint(1, 8)
    alphabet = "abcdef"[:rng.randint(2, 5)]
    run_all(["".join(rng.choice(alphabet) for _ in range(m)) for _ in range(n)])
print("12.4 ngẫu nhiên n <= 5, m <= 8 (1500 ca): OK")

# 12.5 trung bình: n <= 7, m tới 40, so với DP độc lập (và bằng chứng sắp xếp)
rng = random.Random(5955)
for _ in range(120):
    n, m = rng.randint(2, 7), rng.randint(13, 40)
    alphabet = "abcd"[:rng.randint(2, 4)]
    strs = ["".join(rng.choice(alphabet) for _ in range(m)) for _ in range(n)]
    run_all(strs, brute_limit=0)
print("12.5 ngẫu nhiên n <= 7, m tới 40 (120 ca, so với DP): OK")

# 12.6 lớn n = m = 100: các cách khớp nhau; dữ liệu có cấu trúc để có đáp án biết trước
rng = random.Random(6955)
n = m = 100
big = {}
big["đã sắp xếp: đáp án 0"] = (sorted("".join(rng.choice("abc") for _ in range(m)) for _ in range(n)), 0)
big["toàn bằng nhau: 0"] = (["a" * m] * n, 0)
big["giảm dần theo từng cột: m"] = ([chr(ord("z") - i % 26) * m for i in range(n)], None)
big["ngẫu nhiên"] = (["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(m)) for _ in range(n)], None)
for name, (strs, expected) in big.items():
    ref = min_deletion_greedy(strs)
    for fname, fn in solutions_list().items():
        assert fn(list(strs)) == ref, (name, fname)
    d = deleted_indices(strs)
    assert len(d) == ref and is_sorted_after_deleting(strs, d)
    if expected is not None:
        assert ref == expected, (name, ref, expected)
    print(f"  {name:28} đáp án {ref}")
print("12.6 n = m = 100: OK")

# 12.7 tính chất: đáp án <= m; thêm một chuỗi ở cuối không làm giảm đáp án; hoán đổi chữ cái bảo toàn thứ tự (ánh xạ tăng) không đổi đáp án
rng = random.Random(7955)
for _ in range(300):
    n, m = rng.randint(1, 6), rng.randint(1, 8)
    strs = ["".join(rng.choice("abcd") for _ in range(m)) for _ in range(n)]
    base = Solution().minDeletionSize(list(strs))
    assert 0 <= base <= m
    extra = "".join(rng.choice("abcd") for _ in range(m))
    assert Solution().minDeletionSize(strs + [extra]) >= base
    shifted = ["".join(chr(ord(ch) + 3) for ch in s) for s in strs]
    assert Solution().minDeletionSize(shifted) == base
print("12.7 tính chất (<= m, thêm chuỗi, ánh xạ tăng chữ cái): OK")

12.4 ngẫu nhiên n <= 5, m <= 8 (1500 ca): OK
12.5 ngẫu nhiên n <= 7, m tới 40 (120 ca, so với DP): OK
  đã sắp xếp: đáp án 0         đáp án 0
  toàn bằng nhau: 0            đáp án 0
  giảm dần theo từng cột: m    đáp án 100
  ngẫu nhiên                   đáp án 100
12.6 n = m = 100: OK
12.7 tính chất (<= m, thêm chuỗi, ánh xạ tăng chữ cái): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm với bài 944 (mỗi cột phải không giảm)

Bài **944** (Delete Columns to Make Sorted) yêu cầu **từng cột** không giảm từ trên xuống. Bài **955** yêu cầu **các chuỗi** sắp xếp theo thứ tự từ điển, cho phép cột "sai" nếu cặp đã được phân định bởi cột trước.

In [20]:
strs = counters["nhầm với bài 944 (xét từng cột)"]
print("strs =", strs, "| bài 944 ->", wrong_column_sorted(strs), "| đúng cho 955 ->", Solution().minDeletionSize(list(strs)))
assert wrong_column_sorted(strs) != Solution().minDeletionSize(list(strs))
print("ví dụ 2: [xc, yb, za] ->", wrong_column_sorted(["xc", "yb", "za"]), "(944) so với", Solution().minDeletionSize(["xc", "yb", "za"]), "(955)")

strs = ['abaac', 'cbbba', 'cbbbc', 'baacc'] | bài 944 -> 4 | đúng cho 955 -> 3
ví dụ 2: [xc, yb, za] -> 1 (944) so với 0 (955)


Vì sao sai: ở `["xc","yb","za"]` cột `1` (`c, b, a`) giảm, nhưng các chuỗi đã sắp xếp nhờ cột `0`. **Đúng:** theo dõi các cặp đã phân định.

### Lỗi 2 — Kiểm tra mọi cặp ở mọi cột

Không dùng cờ phân định: mọi cột có `strs[i][c] > strs[i + 1][c]` ở **bất kỳ** cặp nào bị xóa, kể cả khi cặp đó đã phân định.

In [21]:
strs = counters["kiểm tra mọi cặp ở mọi cột"]
print("strs =", strs, "| sai ->", wrong_check_all_pairs(list(strs)), "| đúng ->", Solution().minDeletionSize(list(strs)))
assert wrong_check_all_pairs(list(strs)) != Solution().minDeletionSize(list(strs))

strs = ['abaac', 'cbbba', 'cbbbc', 'baacc'] | sai -> 4 | đúng -> 3


Vì sao sai: xóa thừa cột. **Đúng:** chỉ xét các cặp **chưa phân định**.

### Lỗi 3 — Coi cặp bằng nhau là đã phân định

Hai chuỗi bằng nhau trên các cột giữ **chưa** được phân định: cột sau vẫn quyết định thứ tự.

In [22]:
strs = counters["coi cặp bằng nhau là đã phân định"]
print("strs =", strs, "| sai ->", wrong_equal_is_resolved(list(strs)), "| đúng ->", Solution().minDeletionSize(list(strs)))
assert wrong_equal_is_resolved(list(strs)) != Solution().minDeletionSize(list(strs))

strs = ['babb', 'acbc', 'ccaa', 'cccc'] | sai -> 1 | đúng -> 3


Vì sao sai: nếu `strs[i][c] == strs[i+1][c]` thì thứ tự do cột sau quyết định. **Đúng:** chỉ đánh dấu phân định khi `<` nghiêm ngặt.

### Lỗi 4 — Cập nhật cờ trước khi quyết định xóa cột

Một cột bị **xóa** không được phép phân định bất kỳ cặp nào (nó không còn trong chuỗi cuối).

In [23]:
strs = counters["cập nhật cờ trước khi quyết định"]
print("strs =", strs, "| sai ->", wrong_update_before_decide(list(strs)), "| đúng ->", Solution().minDeletionSize(list(strs)))
assert wrong_update_before_decide(list(strs)) != Solution().minDeletionSize(list(strs))

strs = ['babb', 'acbc', 'ccaa', 'cccc'] | sai -> 1 | đúng -> 3


Vì sao sai: cột bị xóa vẫn "phân định" các cặp nên các cột sau bị bỏ qua nhầm. **Đúng:** chỉ cập nhật cờ khi **giữ** cột.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "xóa cột để các hàng sắp xếp theo từ điển" ⇒ **tham lam theo cột** với mảng **cặp đã phân định**.
- **So sánh từ điển** do vị trí khác nhau đầu tiên quyết định; cột sau chỉ quan trọng cho các cặp **chưa phân định**.
- Khi giữ một cột: phân định các cặp có `<`; cặp `=` vẫn chưa.
- Cùng họ: Delete Columns to Make Sorted (944), Delete Columns to Make Sorted III (960, DP), Lexicographically Smallest String After Operations.
- Test nên có: một chuỗi, hai chuỗi đảo thứ tự, cặp bằng nhau, cột "giảm" nhưng đã phân định, `n = m = 100`.

## 15. Biến thể

1. **Trả về các chỉ số cột bị xóa** (bài tập 5): nhiều tập tối ưu có thể tồn tại; tham lam cho một tập.
2. **944:** chỉ đếm cột không giảm: `sum(any(col[i] > col[i+1]) ...)`.
3. **960:** mỗi **hàng** (sau khi xóa) phải không giảm theo ký tự: quy về dãy con không giảm dài nhất của cột (DP).
4. Bài liên quan: 944, 960.

Dưới đây là biến thể 3 (960) kiểm bằng vét cạn.

In [24]:
def min_deletion_960(strs: List[str]) -> int:
    """Mỗi chuỗi sau khi xóa phải không giảm: giữ tập cột dài nhất (dãy con) sao cho mọi hàng không giảm."""
    m = len(strs[0])
    dp = [1] * m
    for j in range(m):
        for i in range(j):
            if all(s[i] <= s[j] for s in strs):
                dp[j] = max(dp[j], dp[i] + 1)
    return m - max(dp)

def min_deletion_960_brute(strs):
    m = len(strs[0])
    best = 0
    for mask in range(1 << m):
        keep = [c for c in range(m) if mask >> c & 1]
        if all(all(s[keep[t]] <= s[keep[t + 1]] for t in range(len(keep) - 1)) for s in strs):
            best = max(best, len(keep))
    return m - best

rng = random.Random(15)
for _ in range(300):
    n, m = rng.randint(1, 4), rng.randint(1, 8)
    strs = ["".join(rng.choice("abc") for _ in range(m)) for _ in range(n)]
    assert min_deletion_960(strs) == min_deletion_960_brute(strs), strs
print("biến thể 960 khớp vét cạn; ví dụ:", min_deletion_960(["babca", "bbazb"]), min_deletion_960(["edcba"]))

biến thể 960 khớp vét cạn; ví dụ: 3 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập cột | Định nghĩa | `O(2^m n m)` | `O(n m)` | Oracle, `m <= 12` |
| 2. Tham lam + cờ | Cặp đã phân định | `O(n m)` | `O(n)` | **Cách nộp chính** |
| 3. Dựng lại chuỗi | Thử thêm cột | `O(n m^2)` | `O(n m)` | Dễ hiểu, chậm hơn |
| 4. DP tập cặp | Không giả định tham lam | mũ theo `n` | mũ theo `n` | Kiểm chứng, `n <= 8` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(list(zip(*["ca", "bb", "ac"])))
print("abd" <= "abc", "ab" <= "abc", "b" <= "ab")
print(min_deletion_brute(["ba", "ab"]))
```

In [25]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `strs = ["abx", "aby", "aab"]`: ghi số cột phải xóa và các chỉ số cột bị xóa (theo tham lam).

In [26]:
# Bài 2: điền kết quả tính tay
hand_answer = None
hand_deleted = None           # danh sách chỉ số cột

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành tham lam với cờ phân định.

In [27]:
def exercise_3_greedy(strs: List[str]) -> int:
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    deleted = 0
    for c in range(m):
        # TODO 1: nếu có cặp chưa phân định i với strs[i][c] > strs[i + 1][c] thì deleted += 1 và continue
        # TODO 2: ngược lại đánh dấu phân định các cặp có strs[i][c] < strs[i + 1][c]
        pass
    return deleted


# for strs, expected in examples:
#     assert exercise_3_greedy(strs) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 1, hãy tự viết `exercise_4_brute(strs)` thử mọi tập cột giữ bằng bitmask.

In [28]:
def exercise_4_brute(strs: List[str]) -> int:
    # TODO
    pass


# for strs, expected in examples:
#     assert exercise_4_brute(strs) == expected

### Bài 5 — Mini challenge: trả về các chỉ số cột bị xóa (khó)

Viết `exercise_5_deleted(strs)` trả về **danh sách chỉ số cột** cần xóa (số lượng đúng bằng đáp án tối thiểu) sao cho các chuỗi còn lại sắp xếp theo từ điển. Có thể có nhiều đáp án; ta kiểm tra bằng hàm kiểm tra.

```python
exercise_5_deleted(["ca", "bb", "ac"])  -> [0]
exercise_5_deleted(["xc", "yb", "za"])  -> []
```

In [29]:
def exercise_5_deleted(strs: List[str]) -> List[int]:
    # TODO
    pass


# for strs, expected in examples:
#     d = exercise_5_deleted(strs)
#     assert len(d) == expected and is_sorted_after_deleting(strs, d)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `zip(*strs)` cho các cột; so sánh chuỗi theo từ điển.
- Mức 2: `"abd" <= "abc"` sai; `"ab" <= "abc"` đúng (tiền tố); `"b" <= "ab"` sai.
- Mức 3: `[('c', 'b', 'a'), ('a', 'b', 'c')]`, `False True False`, `1`.

**Bài 2**
- Mức 1: cột `0` là `a, a, a` (bằng nhau, giữ nhưng chưa phân định cặp nào).
- Mức 2: cột `1` là `b, b, a`: cặp `(1, 2)` chưa phân định có `b > a` ⇒ xóa; cột `2` là `x, y, b`: cặp `(1, 2)` chưa phân định có `y > b` ⇒ xóa.
- Mức 3: `2` cột, chỉ số `[1, 2]`.

**Bài 3**
- Mức 1: `any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1))`.
- Mức 2: nếu đúng thì `deleted += 1; continue`.
- Mức 3: ngược lại `if strs[i][c] < strs[i + 1][c]: resolved[i] = True`.

**Bài 4**
- Mức 1: `for mask in range(1 << m)`.
- Mức 2: các chuỗi rút gọn theo cột có bit `1`.
- Mức 3: giữ `len(keep)` lớn nhất khi `reduced` sắp xếp; trả `m - best`.

**Bài 5**
- Mức 1: như tham lam nhưng lưu chỉ số cột bị xóa.
- Mức 2: `out.append(c)` khi cột vi phạm.
- Mức 3: trả `out`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [30]:
print(list(zip(*["ca", "bb", "ac"])))
print("abd" <= "abc", "ab" <= "abc", "b" <= "ab")
print(min_deletion_brute(["ba", "ab"]))

assert list(zip(*["ca", "bb", "ac"])) == [("c", "b", "a"), ("a", "b", "c")]
assert ("abd" <= "abc", "ab" <= "abc", "b" <= "ab") == (False, True, False)
assert min_deletion_brute(["ba", "ab"]) == 1

[('c', 'b', 'a'), ('a', 'b', 'c')]
False True False
1


### Đáp án Bài 2

Cột `1` và `2` đều phải xóa: `2` cột, chỉ số `[1, 2]`.

In [31]:
strs = ["abx", "aby", "aab"]
hand_answer_solution = 2
hand_deleted_solution = [1, 2]
assert Solution().minDeletionSize(list(strs)) == hand_answer_solution == min_deletion_brute(strs)
assert deleted_indices(strs) == hand_deleted_solution and is_sorted_after_deleting(strs, hand_deleted_solution)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [32]:
def exercise_3_greedy_solution(strs: List[str]) -> int:
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    deleted = 0
    for c in range(m):
        if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
            deleted += 1
            continue
        for i in range(n - 1):
            if strs[i][c] < strs[i + 1][c]:
                resolved[i] = True
    return deleted

for strs, expected in examples:
    assert exercise_3_greedy_solution(strs) == expected
rng = random.Random(3)
for _ in range(300):
    n, m = rng.randint(1, 5), rng.randint(1, 8)
    strs = ["".join(rng.choice("abc") for _ in range(m)) for _ in range(n)]
    assert exercise_3_greedy_solution(strs) == min_deletion_brute(strs)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [33]:
def exercise_4_brute_solution(strs: List[str]) -> int:
    m = len(strs[0])
    best = 0
    for mask in range(1 << m):
        keep = [c for c in range(m) if mask >> c & 1]
        reduced = ["".join(s[c] for c in keep) for s in strs]
        if all(reduced[i] <= reduced[i + 1] for i in range(len(reduced) - 1)):
            best = max(best, len(keep))
    return m - best

for strs, expected in examples:
    assert exercise_4_brute_solution(strs) == expected
rng = random.Random(4)
for _ in range(200):
    n, m = rng.randint(1, 5), rng.randint(1, 8)
    strs = ["".join(rng.choice("abc") for _ in range(m)) for _ in range(n)]
    assert exercise_4_brute_solution(strs) == Solution().minDeletionSize(list(strs))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [34]:
def exercise_5_deleted_solution(strs: List[str]) -> List[int]:
    n, m = len(strs), len(strs[0])
    resolved = [False] * (n - 1)
    out = []
    for c in range(m):
        if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
            out.append(c)
            continue
        for i in range(n - 1):
            if strs[i][c] < strs[i + 1][c]:
                resolved[i] = True
    return out

for strs, expected in examples:
    d = exercise_5_deleted_solution(strs)
    assert len(d) == expected and is_sorted_after_deleting(strs, d)
rng = random.Random(5)
for _ in range(300):
    n, m = rng.randint(1, 5), rng.randint(1, 8)
    strs = ["".join(rng.choice("abc") for _ in range(m)) for _ in range(n)]
    d = exercise_5_deleted_solution(strs)
    assert len(d) == min_deletion_brute(strs) and is_sorted_after_deleting(strs, d)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tham lam theo cột với cờ phân định)

```python
class Solution:
    def minDeletionSize(self, strs: List[str]) -> int:
        n, m = len(strs), len(strs[0])
        resolved = [False] * (n - 1)               # resolved[i]: cặp (i, i + 1) đã phân định (nhỏ hơn hẳn)
        deleted = 0
        for c in range(m):
            if any(not resolved[i] and strs[i][c] > strs[i + 1][c] for i in range(n - 1)):
                deleted += 1                        # giữ cột này sẽ làm strs[i] > strs[i + 1] ở cặp chưa phân định
                continue                            # cột bị xóa KHÔNG cập nhật cờ
            for i in range(n - 1):
                if strs[i][c] < strs[i + 1][c]:     # chỉ '<' nghiêm ngặt mới phân định
                    resolved[i] = True
        return deleted
```

Công thức nhớ nhanh: xóa cột nếu có cặp **chưa phân định** với `>`; giữ cột thì phân định các cặp có `<`.

### Checklist trước khi submit

- [ ] Chỉ xét các cặp **chưa phân định** khi kiểm tra một cột?
- [ ] Phân định chỉ khi `<` nghiêm ngặt (không phải `<=`)?
- [ ] Cột bị xóa **không** cập nhật cờ?
- [ ] Không nhầm với bài 944 (xét từng cột độc lập)?
- [ ] Đã thử: một chuỗi, cặp bằng nhau, cột giảm nhưng đã phân định, `n = m = 100`?
- [ ] Tên method đúng `minDeletionSize(self, strs)` và trả `int`?